# 학습 v2 — QLoRA (train 매칭 데이터, 공격적 설정)

## 규칙 대조 (진행 전 확인)

| 규칙 | 이 작업 | 판정 |
|---|---|---|
| 4.1a 베이스 모델 Qwen2.5-3B-Instruct 고정 | 변경 없음 | ○ |
| 4.2 LoRA/QLoRA/SFT 허용 | QLoRA + SFT | ○ |
| 4.3 다른 모델 가중치 로드/병합 금지 | 없음 | ○ |
| 5.1b test 학습 사용 금지 | test 파일 미보유 | ○ |
| 5.2 공개 데이터 | NuminaMath-1.5 (Apache 2.0) | ○ |
| 5.2c 목록 명시 | **제출 시 기재 필요** | ▲ |

## 지난번(8/19) 학습에서 배운 것

```
최적 체크포인트: checkpoint-900  (전체 906 중)
```
**eval loss가 마지막 step까지 계속 내려갔습니다.** 과적합이 아니라 **학습이 덜 된 상태로 끝난 것**입니다.
어중간한 학습은 잘 조율된 모델을 흔들어놓기만 하고 새 능력은 안 생깁니다. 결과가 -0.67%p였던 이유로 봅니다.

## 이번에 바꾸는 것

| | 8/19 | 이번 | 이유 |
|---|---|---|---|
| 주 데이터 | NuminaMath 무작위 | **train 매칭 12,198** | 대회 문제 그 자체 = 분포 일치 |
| 총량 | 15,000 | **20,000** | 학습량 증가 |
| LoRA rank | 32 | **64** | 표현력 증가 |
| 학습률 | 1e-4 | **2e-4** | 같은 시간에 더 많이 학습 |
| 에폭 | 1 | 1 | 유지 |

## 데이터 우선순위
1. **`train_match` 전부** (12,198) — 이번 가설의 핵심
2. **`rft` 전부** (5,465) — 형식 안정성 (파싱 실패율 42% 감소 기여)
3. `numina_random` — 남는 자리만

## 소요: 약 6~7시간
`[6]` 시작 5분 뒤 ETA 확인 필수.

## 실행 순서
`[1]` `[2]` → ⛔Restart → `[1]` → `[3]`~`[7]`


---
## [1] 설정 ▶️

### ★ 맨 위 두 줄
`CUDA_VISIBLE_DEVICES="0"` — T4가 2장이면 Trainer가 DataParallel을 켜는데,
**4비트 양자화 모델은 복제가 안 됩니다**(양자화 상태가 특정 GPU에 묶여 있음).
8/14에 여기서 `CUDA illegal memory access`로 죽었습니다.

### ★ `LORA_R=64`, `LR=2e-4`
지난번이 학습 부족이었으므로 둘 다 올립니다.
rank 64면 학습 파라미터가 약 1.2억 개(전체의 3.7%)로 두 배가 됩니다.

⚠️ `LR`을 올리면 fp16에서 불안정해질 수 있습니다. loss가 `nan`이면 즉시 중단하고 `1.5e-4`로 낮추세요.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # ★ 4bit + DataParallel 충돌 방지. 반드시 맨 위

# ── 데이터 ────────────────────────────────────────────
N_TRAIN    = 20000     # 우선순위대로 채움
MAX_LEN    = 1024      # 초과 샘플은 자르지 않고 버림
EVAL_RATIO = 0.01

# ── LoRA (★ 지난번보다 공격적) ─────────────────────────
LORA_R     = 64        # 32 → 64
LORA_ALPHA = 128       # r의 2배
LORA_DROP  = 0.05

# ── 학습 ──────────────────────────────────────────────
EPOCHS     = 1
LR         = 2e-4      # 1e-4 → 2e-4
BATCH      = 4         # OOM이면 2 (GRAD_ACC는 8)
GRAD_ACC   = 4
SEED       = 42

MODEL_ID   = "Qwen/Qwen2.5-3B-Instruct"
OUT_DIR    = "/kaggle/working/qwen25-3b-v2-lora"
SYSTEM = ("You are an expert competition mathematician. Solve the problem step by step, "
          "concisely. The final answer is ALWAYS a single integer. "
          "End your response with the final integer inside \\boxed{}.")
# ──────────────────────────────────────────────────────
print(f"N_TRAIN={N_TRAIN} | r={LORA_R} | lr={LR} | 실질배치={BATCH*GRAD_ACC} | 예상 step≈{N_TRAIN//(BATCH*GRAD_ACC)}")

---
## [2] 설치 ⏭️ 세션 안 껐으면 건너뛰기
---
## ⛔ Restart Session → [1]부터
---

In [ ]:
!pip install -q -U peft bitsandbytes accelerate 2>&1 | tail -3
import torch, peft, bitsandbytes, transformers
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("peft        :", peft.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("보이는 GPU  :", torch.cuda.device_count(), "(1이어야 함)")

---
## [3] 데이터 로드 (우선순위 방식) ▶️

무작위로 20,000개를 뽑으면 `train_match` 비중이 41%밖에 안 됩니다.
**우선순위대로 채워서** 핵심 데이터를 전부 넣습니다.

In [ ]:
import json, glob, random
from collections import Counter

paths = [p for p in glob.glob("/kaggle/input/**/*.jsonl", recursive=True)
         if "v2" in os.path.basename(p).lower()] or \
        glob.glob("/kaggle/input/**/*.jsonl", recursive=True)
print("찾은 jsonl:", paths)
assert paths, "train_v2.jsonl 을 못 찾았습니다. + Add Input 으로 Dataset을 추가하세요."

data = [json.loads(l) for l in open(paths[0], encoding="utf-8")]
print(f"\n원본 {len(data):,}  {dict(Counter(r['src'] for r in data))}")

by = {k: [r for r in data if r["src"] == k] for k in ["train_match", "rft", "numina_random"]}
for v in by.values():
    random.Random(SEED).shuffle(v)

sel = []
for k in ["train_match", "rft", "numina_random"]:          # ★ 우선순위
    take = by[k][:max(0, N_TRAIN - len(sel))]
    sel += take
    print(f"  {k:<14} {len(take):>6,} 개 사용 / 보유 {len(by[k]):,}")

random.Random(SEED).shuffle(sel)
data = sel
print(f"\n선택 {len(data):,}  {dict(Counter(r['src'] for r in data))}")

---
## [4] 토큰화 + 손실 마스킹 ▶️

프롬프트 구간 라벨을 `-100`으로 채웁니다(PyTorch에서 "손실 제외").
모델이 배워야 할 건 풀이 쓰는 법이지 문제 지어내는 법이 아닙니다.

`MAX_LEN` 초과 샘플은 **자르지 않고 버립니다.** 잘린 풀이엔 `\boxed{}`가 없어
"답 없이 끝내는 법"을 학습시키게 됩니다.

In [ ]:
import torch, pandas as pd
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
print("eos:", repr(tok.eos_token), "| pad:", repr(tok.pad_token))

def build(r):
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user",   "content": r["question"]}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    full   = tok.apply_chat_template(
        msgs + [{"role": "assistant", "content": r["solution"]}], tokenize=False)
    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    f_ids = tok(full,   add_special_tokens=False)["input_ids"]
    if len(f_ids) > MAX_LEN:
        return None
    labels = list(f_ids)
    for i in range(min(len(p_ids), len(labels))):
        labels[i] = -100
    return {"input_ids": f_ids, "labels": labels}

built = [(build(r), r["src"]) for r in data]
ds = [b for b, s in built if b is not None]
drop = len(built) - len(ds)
print(f"\nMAX_LEN 초과 제외: {drop:,} ({drop/len(built):.1%})")
print("남은 src:", dict(Counter(s for b, s in built if b is not None)))

lens = pd.Series([len(e["input_ids"]) for e in ds])
print(f"토큰 길이 중앙값 {lens.median():.0f} / p95 {lens.quantile(.95):.0f}")

n_eval = max(80, int(len(ds) * EVAL_RATIO))
eval_ds, train_ds = ds[:n_eval], ds[n_eval:]
tot = lens.sum() * EPOCHS
print(f"\n학습 {len(train_ds):,} / 검증 {len(eval_ds):,}")
print(f"총 토큰 {tot/1e6:.1f}M  (8/19 6.2M → {tot/6.2e6:.2f}배)")
print(f"단순 환산 예상 시간: 약 {tot/1e6/1.37:.1f}시간 (rank 64 오버헤드 별도)")

---
## [5] 4비트 모델 + LoRA(rank 64) ▶️

확인할 것:
- `trainable params` 약 **1.2억 개(3.7%)** ← rank 64라 지난번(5,987만/1.90%)의 두 배
- `GPU 사용` **2.9~3.1GB** ← 4비트 양자화 성공

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # T4는 bf16 불가
    bnb_4bit_use_double_quant=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map={"": 0}, trust_remote_code=True)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROP,
    bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"]))
model.print_trainable_parameters()
print(f"\nGPU 사용: {torch.cuda.memory_allocated()/1e9:.2f} GB")

---
## [6] 학습 ▶️ ★ 시작 5분 뒤 ETA 확인

### ETA 판단

| ETA | 조치 |
|---|---|
| 7시간 이하 | 그대로 진행 |
| 7~9시간 | 진행하되 세션 12시간 제한 주의 |
| **9시간 초과** | **중단 → `N_TRAIN=15000`** |

### 진행 중 볼 것
- **`nan`이 나오면 즉시 중단** → `LR=1.5e-4`로 재시도. `LR`을 두 배로 올렸으므로 위험이 있습니다
- 8/19에는 eval loss가 0.4006에서 시작해 끝까지 하락했습니다.
  이번엔 **train 매칭 데이터라 시작값이 더 낮을 수** 있습니다(모델이 이미 본 유형)
- eval이 중간에 올라가면 `load_best_model_at_end`가 최저 시점으로 자동 복원합니다

In [ ]:
from transformers import Trainer, TrainingArguments

def collate(batch):
    m = max(len(b["input_ids"]) for b in batch)
    out = {"input_ids": [], "labels": [], "attention_mask": []}
    for b in batch:
        pad = m - len(b["input_ids"])
        out["input_ids"].append(b["input_ids"] + [tok.pad_token_id] * pad)
        out["labels"].append(b["labels"] + [-100] * pad)
        out["attention_mask"].append([1] * len(b["input_ids"]) + [0] * pad)
    return {k: torch.tensor(v) for k, v in out.items()}

args = TrainingArguments(
    output_dir="/kaggle/working/ckpt",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_grad_norm=0.3,
    fp16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=25,
    eval_strategy="steps",   eval_steps=100,
    save_strategy="steps",   save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    per_device_eval_batch_size=BATCH,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(model=model, args=args,
                  train_dataset=train_ds, eval_dataset=eval_ds,
                  data_collator=collate)
trainer.train()

---
## [7] 어댑터 저장 ▶️

rank 64라 지난번(224MB)보다 큽니다. **약 440MB** 예상.

### 저장 후
1. **zip 다운로드** (세션 끄기 전!)
2. **Create → New Dataset** → 이름 `qwen25-3b-v2-lora`
3. A/B 검증 시 ⚠️ **어댑터 Dataset은 이것 하나만** Input에 붙일 것
   (rft·numina 어댑터가 같이 붙으면 자동 탐색이 엉뚱한 걸 잡습니다)
4. A/B 노트북 `[1]`에서 **`LORA_RANK = 64`로 변경** — 안 바꾸면 vLLM이 어댑터를 못 읽습니다

In [ ]:
import shutil

print("최적 체크포인트:", trainer.state.best_model_checkpoint)
print("최저 eval loss :", trainer.state.best_metric)

model.save_pretrained(OUT_DIR)
tok.save_pretrained(OUT_DIR)
for fn in sorted(os.listdir(OUT_DIR)):
    print(f"  {fn}  ({os.path.getsize(os.path.join(OUT_DIR,fn))/1e6:.1f} MB)")

zp = shutil.make_archive("/kaggle/working/v2_lora", "zip", OUT_DIR)
print(f"\nzip: {zp} ({os.path.getsize(zp)/1e6:.1f} MB)")
shutil.rmtree("/kaggle/working/ckpt", ignore_errors=True)

print("\n★ A/B 검증 시 LORA_RANK = 64 로 바꾸는 것 잊지 마세요")

from IPython.display import FileLink
FileLink("v2_lora.zip")